# 05 - Churn Prediction\nAll modelling steps live in `src/models.py`. Preprocessing sits inside each sklearn Pipeline, so scaling, encoding, resampling and clustering are fitted on training folds only.

## 1. Train/test split
Stratified 80/20 split on the churn label, so both halves keep the 26.5% churn rate (5,634 train / 1,409 test). The **test set is touched once**, at the end. All model selection uses 5-fold stratified cross-validation on the training set. `TotalCharges` is dropped because it is roughly `tenure x MonthlyCharges` (redundant, and it muddies coefficients); `customerID` is dropped as an identifier.

In [1]:
import sys, warnings; sys.path.append('..'); warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, accuracy_score
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from src import eda, models as M
pd.set_option('display.width', 200); sns.set_theme(style='whitegrid')
df = eda.load(); X, y = M.split_xy(df)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=M.SEED)
print('train', Xtr.shape, 'test', Xte.shape, '| churn rate train/test:', ytr.mean().round(4), yte.mean().round(4))
print('features:', list(X.columns))

train (5634, 18) test (1409, 18) | churn rate train/test: 0.2654 0.2654
features: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges']


## 2. Why accuracy is the wrong metric here

In [2]:
print('Accuracy of "predict nobody churns" on test:', round(1 - yte.mean(), 4))
print('...and its recall on churners:', 0.0)

Accuracy of "predict nobody churns" on test: 0.7346
...and its recall on churners: 0.0


## 3. Model x imbalance-strategy comparison (5-fold CV on train)
Three models (logistic regression baseline, random forest, gradient boosting) crossed with three ways of handling the 2.8:1 imbalance (nothing, class weights, SMOTE inside the CV fold). Metrics are ROC-AUC and PR-AUC (average precision), which judge the *ranking* of customers and do not depend on a threshold.

In [3]:
rows = []
for name in ['logreg', 'rf', 'hgb']:
    for imb in ['none', 'class_weight', 'smote']:
        s = M.cv_scores(M.make_model(name, Xtr, imb), Xtr, ytr)
        rows.append({'model': name, 'imbalance': imb,
                     'ROC-AUC': f"{s['roc_auc'][0]:.3f} +/- {s['roc_auc'][1]:.3f}",
                     'PR-AUC': f"{s['pr_auc'][0]:.3f} +/- {s['pr_auc'][1]:.3f}"})
grid = pd.DataFrame(rows); grid

,model,imbalance,ROC-AUC,PR-AUC
0,logreg,none,0.845 +/- 0.012,0.660 +/- 0.019
1,logreg,class_weight,0.845 +/- 0.012,0.659 +/- 0.019
2,logreg,smote,0.843 +/- 0.012,0.656 +/- 0.022
3,rf,none,0.844 +/- 0.009,0.654 +/- 0.020
4,rf,class_weight,0.844 +/- 0.009,0.650 +/- 0.021
5,rf,smote,0.840 +/- 0.009,0.632 +/- 0.025
6,hgb,none,0.845 +/- 0.012,0.664 +/- 0.019
7,hgb,class_weight,0.845 +/- 0.011,0.664 +/- 0.018
8,hgb,smote,0.844 +/- 0.011,0.658 +/- 0.020


**Findings.**
- All nine configurations land at **ROC-AUC 0.840-0.845**, and the gaps are smaller than one fold-to-fold standard deviation (~0.01). On this dataset the signal is mostly linear and captured by a few features, so model complexity buys almost nothing.
- **SMOTE never helped** and slightly hurt the random forest's PR-AUC (0.632 vs 0.654), because it adds synthetic customers without adding information. Class weights are neutral for ranking, are free, and avoid inventing data.
- **Decision:** logistic regression with `class_weight='balanced'` as the primary model (equal ranking quality, directly interpretable), with gradient boosting kept as a challenger. Note that imbalance handling mainly changes *where the decision threshold sits*, which is why I tune the threshold explicitly below rather than using 0.5.
- SMOTE here is applied after one-hot encoding, so synthetic points on binary columns are fractional. SMOTENC would be stricter, but it did not help even in this lenient form.

## 4. Leakage check: do the Phase 4 segments help prediction?
Three variants of the same model: no segment feature; segment assigned by clustering fitted **inside each training fold** (the leak-safe way, via `SegmentFeature`); and segments taken from `segments.csv`, which were fitted on **all** rows including validation rows (the leaky way).

In [4]:
from sklearn.pipeline import Pipeline
seg_all = pd.read_csv('../data/segments.csv').set_index(df.customerID)['segment']
Xleak = Xtr.assign(segment_id=seg_all.reindex(df.loc[Xtr.index, 'customerID']).values)   # clusters fit on ALL rows
rows = []
for name in ['logreg', 'hgb']:
    base = M.cv_scores(M.make_model(name, Xtr, 'class_weight'), Xtr, ytr)
    safe = M.cv_scores(M.make_model(name, Xtr, 'class_weight', with_segment=True), Xtr, ytr)
    from sklearn.linear_model import LogisticRegression
    leaky_pipe = Pipeline([('prep', M.preprocessor(Xleak, scale=(name=='logreg'))), ('clf', M.make_model(name, Xtr, 'class_weight').steps[-1][1])])
    leak = M.cv_scores(leaky_pipe, Xleak, ytr)
    rows.append({'model': name, 'no segment': round(base['roc_auc'][0], 4), 'segment fit inside fold': round(safe['roc_auc'][0], 4),
                 'segment fit on all rows (leaky)': round(leak['roc_auc'][0], 4)})
pd.DataFrame(rows)

,model,no segment,segment fit inside fold,segment fit on all rows (leaky)
0,logreg,0.8447,0.8451,0.8451
1,hgb,0.8451,0.8449,0.8453


**Result:** no variant moves AUC beyond the 4th decimal. That is expected: the segments are a deterministic function of the same features the model already sees, so they add no new information. They remain valuable for *business interpretation* (Phase 4) but not for prediction. The leaky variant also scores the same here because clustering is unsupervised and never sees the label; the risk would be real for any target-aware feature (target encoding, supervised binning). I keep the fold-safe `SegmentFeature` in the codebase as the default pattern and leave segments out of the final model for simplicity.

## 5. Final models on the held-out test set
The decision threshold is chosen by maximising F1 on **out-of-fold predictions from the training set**, never on the test set. Then each model is fit on the full training set and scored on the test set once.

In [5]:
final = {}
for name in ['logreg', 'hgb']:
    model = M.make_model(name, Xtr, 'class_weight')
    oof = M.oof_proba(model, Xtr, ytr)
    t = M.best_f1_threshold(ytr, oof)
    model.fit(Xtr, ytr)
    p = model.predict_proba(Xte)[:, 1]
    final[name] = dict(model=model, thr=t, p=p)
rows = []
for name, d in final.items():
    m = M.threshold_metrics(yte, d['p'], d['thr'])
    rows.append({'model': name, 'ROC-AUC': round(roc_auc_score(yte, d['p']), 4), 'PR-AUC': round(average_precision_score(yte, d['p']), 4),
                 'threshold (chosen on train OOF)': m['threshold'], 'precision': round(m['precision'], 3), 'recall': round(m['recall'], 3),
                 'F1': round(m['f1'], 3), 'accuracy': round(accuracy_score(yte, d['p'] >= d['thr']), 3), 'TP': m['tp'], 'FP': m['fp'], 'FN': m['fn'], 'TN': m['tn']})
pd.DataFrame(rows)

,model,ROC-AUC,PR-AUC,threshold (chosen on train OOF),precision,recall,F1,accuracy,TP,FP,FN,TN
0,logreg,0.8387,0.6283,0.55,0.524,0.743,0.614,0.752,278,253,96,782
1,hgb,0.8397,0.6589,0.57,0.539,0.735,0.622,0.763,275,235,99,800


**Reading the table.**
- Test ROC-AUC is **0.839 (logistic regression)** and **0.840 (gradient boosting)**, in line with the 0.845 cross-validation estimate, so there is no sign of overfitting to the training folds.
- At the chosen threshold the logistic model catches **74% of churners (recall)** with **52% precision** (about 1 true churner per 2 customers flagged). Accuracy is only **75.2%**, barely above the 73.5% do-nothing baseline, which is exactly why accuracy would have hidden how much better the model is: that baseline catches **0%** of churners.
- Gradient boosting has a higher PR-AUC on test (0.659 vs 0.628), while cross-validation showed a much smaller gap (0.664 vs 0.659). Notebook 06 tests whether the test-set gap is real with a paired bootstrap.
- **Precision is the cost of recall.** A retention team that tolerates ~1 wasted offer per useful one gets 74% coverage; a team with a tight budget should use a higher threshold or the top-decile list below.

In [6]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for name, d in final.items():
    fpr, tpr, _ = roc_curve(yte, d['p']); ax[0].plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(yte, d['p']):.3f})")
    pr, rc, _ = precision_recall_curve(yte, d['p']); ax[1].plot(rc, pr, label=f"{name} (AP {average_precision_score(yte, d['p']):.3f})")
ax[0].plot([0, 1], [0, 1], 'k--', lw=1); ax[0].set(title='ROC (test set)', xlabel='False positive rate', ylabel='True positive rate')
ax[1].axhline(yte.mean(), c='k', ls='--', lw=1); ax[1].set(title='Precision-recall (test set)', xlabel='Recall', ylabel='Precision')
ax[0].legend(); ax[1].legend(); plt.tight_layout(); plt.savefig('../reports/figures/09_roc_pr.png', dpi=150); plt.show()

## 6. Business view: lift and revenue captured
Ranking customers by predicted risk and contacting only the top slice. Decile 1 is the 10% of customers the model considers riskiest.

In [7]:
rev = df.loc[Xte.index, 'MonthlyCharges'].values
lt = M.lift_table(yte, final['logreg']['p'], revenue=rev)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(lt.index, lt.churn_rate * 100, color='#E4572E'); ax.axhline(yte.mean() * 100, ls='--', c='grey')
for i, v in zip(lt.index, lt.lift): ax.text(i, lt.churn_rate[i] * 100 + 1, f'{v:.1f}x', ha='center', fontsize=9)
ax.set(xlabel='Risk decile (1 = highest predicted risk)', ylabel='Actual churn rate (%)', title='Logistic regression: churn rate and lift by risk decile (test set)', xticks=range(1, 11))
plt.tight_layout(); plt.savefig('../reports/figures/10_lift.png', dpi=150); plt.show()
lt.round(3)

,customers,churners,churn_rate,lift,cum_churners_captured,cum_lost_mrr_captured
decile,,,,,,
1,141,106,0.752,2.832,0.283,0.328
2,141,82,0.582,2.191,0.503,0.561
3,141,51,0.362,1.363,0.639,0.689
4,141,49,0.348,1.309,0.770,0.810
5,141,38,0.270,1.015,0.872,0.887
6,141,24,0.170,0.641,0.936,0.934
7,141,13,0.092,0.347,0.971,0.967
8,141,8,0.057,0.214,0.992,0.992
9,141,1,0.007,0.027,0.995,0.994


In [8]:
# Business comparison: contact the top 20% (20% of the test set) by score vs 20% at random
n = len(yte); k = int(0.2 * n)
rng = np.random.default_rng(0)
order = np.argsort(-final['logreg']['p'])
model_caught = yte[order[:k]].sum()
rand_caught = np.mean([yte[rng.permutation(n)[:k]].sum() for _ in range(2000)])
lost_mrr = (rev * yte)
print(f"contact {k} customers (20%)")
print(f"churners reached  model: {model_caught}/{yte.sum()} ({model_caught/yte.sum():.1%}) | random: {rand_caught:.0f} ({rand_caught/yte.sum():.1%})")
print(f"precision         model: {model_caught/k:.1%} | random: {yte.mean():.1%}")
print(f"churned MRR reached  model: {lost_mrr[order[:k]].sum()/lost_mrr.sum():.1%} | random: ~20%")
print(f"lift in top 20%: {(model_caught/k)/yte.mean():.2f}x")

contact 281 customers (20%)
churners reached  model: 187/374 (50.0%) | random: 75 (20.0%)
precision         model: 66.5% | random: 26.5%
churned MRR reached  model: 55.8% | random: ~20%
lift in top 20%: 2.51x


**Interpretation.**
- The riskiest decile churns at **75%** (**2.8x** the base rate) and the top two deciles together reach **50% of all churners** while being 20% of customers.
- Contacting the top 20% finds **187 of 374 churners (50%) versus ~75 (20%) by random targeting**: **precision 66.5% vs 26.5%**, a **2.5x lift**. Because high-risk customers also pay more, the same list reaches **55.8% of churned monthly revenue**.
- The bottom three deciles hold only 1-3% of churners, so they need no retention spend.
- Caveat: these are *associations on a historical snapshot*. The lift measures how well the model ranks past churners, not how many would be saved by an offer. That needs an A/B test.

## 7. What drives churn?
Odds ratios from the logistic regression (numeric features standardised, so the tenure ratio is per 1 standard deviation). Categorical levels are all kept with L2 regularisation, so each ratio is relative to the model's overall baseline rather than a single reference level.

In [9]:
lr = final['logreg']['model']
names = lr.named_steps['prep'].get_feature_names_out()
coef = pd.Series(lr.named_steps['clf'].coef_[0], index=[n.split('__')[1] for n in names])
orr = np.exp(coef).sort_values()
sel = pd.concat([orr.head(8), orr.tail(8)])
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(sel.index, sel.values - 1, left=1, color=['#4C78A8' if v < 1 else '#E4572E' for v in sel.values])
ax.axvline(1, c='k', lw=1); ax.set(xlabel='Odds ratio (numerics per 1 SD)', title='Logistic regression: strongest effects on churn odds')
plt.tight_layout(); plt.savefig('../reports/figures/11_odds_ratios.png', dpi=150); plt.show()
orr.sort_values(ascending=False).round(2).head(8), orr.sort_values().round(2).head(8)

(InternetService_Fiber optic       2.38
 Contract_Month-to-month           1.72
 StreamingMovies_Yes               1.45
 StreamingTV_Yes                   1.42
 PaperlessBilling_Yes              1.40
 MultipleLines_Yes                 1.37
 SeniorCitizen_1                   1.16
 PaymentMethod_Electronic check    1.16
 dtype: float64,
 InternetService_No                         0.30
 Contract_Two year                          0.41
 tenure                                     0.47
 OnlineSecurity_Yes                         0.70
 TechSupport_Yes                            0.75
 PhoneService_Yes                           0.76
 PaymentMethod_Bank transfer (automatic)    0.77
 PaymentMethod_Credit card (automatic)      0.77
 dtype: float64)

**Reading the coefficients.**
- Raising risk: **fiber optic (2.4x odds)**, **month-to-month contract (1.7x)**, streaming add-ons, paperless billing, multiple lines, electronic-check payment.
- Lowering risk: **no internet service (0.30x)**, **two-year contract (0.41x)**, **longer tenure (0.47x per SD)**, online security and tech support (~0.7x).
- These are *conditional* effects, not causal ones. **Do not trust the streaming coefficients:** `MonthlyCharges` is 99.9% explained by the service columns (shown in notebook 06), so in this model the streaming flags soak up collinear price information. In the Cox model without `MonthlyCharges`, streaming has no effect. L2 regularisation keeps predictions stable despite the collinearity, but individual coefficients are unreliable.
- The signs match the Phase 2-4 findings, which is a useful consistency check.

In [10]:
pi = permutation_importance(final['hgb']['model'], Xte, yte, scoring='roc_auc', n_repeats=20, random_state=M.SEED)
imp = pd.DataFrame({'feature': Xte.columns, 'auc_drop': pi.importances_mean, 'sd': pi.importances_std}).sort_values('auc_drop', ascending=False)
fig, ax = plt.subplots(figsize=(7, 4.5)); top = imp.head(8)[::-1]
ax.barh(top.feature, top.auc_drop, xerr=top.sd, color='#4C78A8'); ax.set(xlabel='Drop in test ROC-AUC when shuffled', title='Gradient boosting: permutation importance')
plt.tight_layout(); plt.savefig('../reports/figures/12_permutation_importance.png', dpi=150); plt.show()
imp.head(8).round(4)

,feature,auc_drop,sd
14,Contract,0.0999,0.0077
4,tenure,0.0760,0.0057
7,InternetService,0.0362,0.0065
16,PaymentMethod,0.0045,0.0016
11,TechSupport,0.0025,0.0018
8,OnlineSecurity,0.0022,0.0010
15,PaperlessBilling,0.0021,0.0012
13,StreamingMovies,0.0019,0.0006


**Permutation importance** (how much test AUC drops when one feature is shuffled) agrees with the coefficients from a model of a different type: **Contract (-0.100 AUC), tenure (-0.076) and InternetService (-0.036)** dominate, and everything else contributes under 0.005 each. Retention strategy should focus on contract terms, early-tenure experience and the fiber product.

In [11]:
out = pd.DataFrame({'customerID': df.loc[Xte.index, 'customerID'].values, 'churn': yte,
    'p_logreg': final['logreg']['p'], 'p_hgb': final['hgb']['p'], 'MonthlyCharges': rev})
out.to_csv('../data/test_predictions.csv', index=False)
joblib.dump(final['logreg']['model'], '../models/logreg_churn.joblib')
pd.Series({n: d['thr'] for n, d in final.items()}).to_json('../models/thresholds.json')
out.shape

(1409, 5)

## Takeaways
- **ROC-AUC ~0.84** on held-out data; imbalance handling and model complexity barely matter on this dataset.
- **Top-20% targeting reaches 50% of churners at 2.5x lift** and 56% of churned revenue.
- Contract, tenure and internet service drive predictions, consistent with the SQL, EDA and clustering findings.
- **Limitations:** single snapshot, no causal claim, no signup dates (so no true time-based validation), and the threshold should be re-tuned to the retention team's actual offer cost.